# Capa Silver Municipal

Este notebook documenta la curacion Silver para el analisis municipal del presupuesto y ejecucion de ingresos.

```mermaid
flowchart LR
  B[Bronze Parquet] --> S[Silver Pipeline]
  S --> D[dim_municipalidad]
  S --> P[fact_predial_esat]
  S --> R[fact_sismepre_respuestas]
  S --> C[Catalogos SISMEPRE]
  S --> Q[_quarantine]
  S --> I[fact_ingresos_municipales]
```


In [1]:
import json
from pathlib import Path
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('silver-notebook-evidence').getOrCreate()
silver_path = Path('../data/silver') if Path('../data/silver').exists() else Path('data/silver')
audit_path = Path('../data/audit') if Path('../data/audit').exists() else Path('data/audit')
summary_files = sorted(audit_path.glob('metrics/**/silver_summary_*.json'))
assert summary_files, 'Ejecute python main_silver.py antes de abrir la evidencia Silver.'
summary = json.loads(summary_files[-1].read_text(encoding='utf-8'))
print('Resumen auditado:', summary_files[-1])
status = 'partial' if summary['blocked_tables'] else ('failed' if summary['errors'] else 'success')
print('Estado:', status)


Resumen auditado: ../data/audit/metrics/2026/06/01/silver_summary_20260601_084205.json
Estado: success


## Inventario publicado y bloqueado

`fact_ingresos_municipales` se publica con registros `NIVEL_GOBIERNO = 'M'`. Si una futura descarga no contiene filas municipales, Silver bloquea solo esta tabla; los registros regionales no se usan como sustituto.

In [2]:
published = [(item['table_name'], item['records_published'], item['records_quarantined']) for item in summary['published']]
blocked = [(item['table_name'], item['reason']) for item in summary['blocked']]
print('PUBLICADAS')
for row in published: print(row)
print('\nBLOQUEADAS')
for row in blocked: print(row)
income_fact = silver_path / 'fact_ingresos_municipales'
if blocked:
    assert not income_fact.exists(), 'No debe existir una fact municipal obsoleta cuando la tabla esta bloqueada.'
else:
    assert income_fact.exists(), 'Debe existir fact_ingresos_municipales cuando la publicacion es exitosa.'


PUBLICADAS
('dim_municipalidad', 1111, 0)
('fact_ingresos_municipales', 8880655, 0)
('fact_predial_esat', 133172, 0)
('fact_sismepre_respuestas', 205823, 2)
('dim_sismepre_entidad_estado', 19037, 0)
('dim_sismepre_pregunta', 696, 0)
('dim_sismepre_formulario', 94, 0)

BLOQUEADAS


## Esquemas Silver

Las tablas curadas conservan trazabilidad Bronze y agregan `_silver_execution_id` y `_silver_ingestion_ts`.

In [3]:
for table_name, _, _ in published:
    print('\n===', table_name, '===')
    frame = spark.read.parquet(str(silver_path / table_name))
    print('rows =', frame.count())
    frame.printSchema()
    assert '_silver_execution_id' in frame.columns
    assert '_silver_ingestion_ts' in frame.columns



=== dim_municipalidad ===
rows = 1111
root
 |-- UBIGEO: string (nullable = true)
 |-- SEC_EJEC: string (nullable = true)
 |-- DEPARTAMENTO_NOMBRE: string (nullable = true)
 |-- PROVINCIA_NOMBRE: string (nullable = true)
 |-- DISTRITO_NOMBRE: string (nullable = true)
 |-- MUNICIPALIDAD_NOMBRE: string (nullable = true)
 |-- _bronze_source_path: string (nullable = true)
 |-- _bronze_source_url: string (nullable = true)
 |-- _bronze_source_checksum: string (nullable = true)
 |-- _bronze_execution_id: string (nullable = true)
 |-- _bronze_ingestion_ts: timestamp (nullable = true)
 |-- _bronze_ingestion_date: date (nullable = true)
 |-- idmunici: string (nullable = true)
 |-- Tipomuni: string (nullable = true)
 |-- RENAMU_DEPARTAMENTO: string (nullable = true)
 |-- RENAMU_PROVINCIA: string (nullable = true)
 |-- RENAMU_DISTRITO: string (nullable = true)
 |-- renamu_match: boolean (nullable = true)
 |-- _silver_execution_id: string (nullable = true)
 |-- _silver_ingestion_ts: timestamp (null

## Calidad documentada

Silver registra completitud, unicidad, validez, consistencia, integridad, actualidad, disponibilidad y exactitud en `data/audit/quality_checks` y en el snapshot de metricas.

In [4]:
quality_rows = []
for item in summary['published']:
    for check in item['quality_checks']:
        quality_rows.append((item['table_name'], check['check_type'], check['status'], check['details']['score']))
for row in quality_rows: print(row)
print('\nchecks =', len(quality_rows), 'failed =', sum(status == 'failed' for _, _, status, _ in quality_rows))


('dim_municipalidad', 'completitud', 'passed', 1.0)
('dim_municipalidad', 'unicidad', 'passed', 1.0)
('dim_municipalidad', 'validez', 'passed', 1.0)
('dim_municipalidad', 'consistencia', 'passed', 1.0)
('dim_municipalidad', 'integridad', 'passed', 1.0)
('dim_municipalidad', 'actualidad', 'passed', 1.0)
('dim_municipalidad', 'disponibilidad', 'passed', 1.0)
('dim_municipalidad', 'exactitud', 'passed', 1.0)
('fact_ingresos_municipales', 'completitud', 'passed', 1.0)
('fact_ingresos_municipales', 'unicidad', 'passed', 1.0)
('fact_ingresos_municipales', 'validez', 'passed', 1.0)
('fact_ingresos_municipales', 'consistencia', 'passed', 1.0)
('fact_ingresos_municipales', 'integridad', 'passed', 1.0)
('fact_ingresos_municipales', 'actualidad', 'passed', 1.0)
('fact_ingresos_municipales', 'disponibilidad', 'passed', 1.0)
('fact_ingresos_municipales', 'exactitud', 'passed', 1.0)
('fact_predial_esat', 'completitud', 'passed', 1.0)
('fact_predial_esat', 'unicidad', 'passed', 1.0)
('fact_predial_es

## Evidencia de cuarentena

Los rechazos se escriben por tabla bajo `data/silver/_quarantine`. Cuando una ejecucion no tiene rechazos, la salida antigua se elimina para no dejar evidencia obsoleta.

In [5]:
quarantine_root = silver_path / '_quarantine'
quarantine_tables = sorted(path.name for path in quarantine_root.iterdir() if path.is_dir()) if quarantine_root.exists() else []
print('Tablas con cuarentena:', quarantine_tables)
for table_name in quarantine_tables:
    frame = spark.read.parquet(str(quarantine_root / table_name))
    print(table_name, 'rows =', frame.count())
    frame.select('_quarantine_reason').groupBy('_quarantine_reason').count().show(truncate=False)


Tablas con cuarentena: ['fact_sismepre_respuestas']
fact_sismepre_respuestas rows = 2
+------------------+-----+
|_quarantine_reason|count|
+------------------+-----+
|no_active_response|2    |
+------------------+-----+



## Preparacion para Gold

Con estas tablas Silver se pueden construir seis productos analiticos sin reprocesar Bronze:

1. Evolucion mensual del presupuesto y recaudacion municipal, cuando Ingresos Bronze incluya nivel `M`.
2. Avance de ejecucion por municipalidad.
3. Ranking territorial por recaudacion.
4. Indicadores de impuesto predial.
5. Cobertura y cumplimiento de metas SISMEPRE.
6. Calidad y cobertura de datos por fuente.
